# Notebook 10: Capstone - ML System Design Review

## Why This Matters

The ML system design interview is a high-stakes, open-ended exercise where you have 45-60 minutes to design a production ML system from scratch. The most common question at senior+ levels is a large-scale recommendation or ranking system. 'Design YouTube recommendations from scratch' is canonical - it appears at Google, Meta, Netflix, Spotify, and many others. This capstone walks through a complete, interview-ready answer with real numbers, design decisions, and justifications. It also covers the 10 most common ML system design interview topics and the most common mistakes candidates make.

In [ ]:
%pip install -q numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("Ready.")

## 1. Problem Framing: YouTube Recommendations

**Business context**: 2.7B monthly active users, 500 hours of video uploaded per minute, 1B hours watched per day.

**Business objective**: Maximize long-term user satisfaction and retention (DAU, subscriber growth).

**Why NOT just maximize watch time?** Watch time is a proxy, not the objective. Users can watch things out of habit without being satisfied. YouTube acknowledged watch time optimization created clickbait/rabbit-hole problems.

**ML objective**: For retrieval - maximize P(user engages with video). For ranking - rank videos by expected satisfaction score.

**Key constraints**:
- Latency: <200ms end-to-end
- Scale: 2B users x 800M videos; retrieval must use ANN
- Cold start: 500 hours/minute new content; new users daily
- Freshness: trending content must surface within hours

**Anti-pattern to avoid**: 'I'll use a transformer that scores every video.' At 800M videos, that's computationally impossible. Start with the scale constraint and derive the architecture.

In [ ]:
design_numbers = {
    "Users (MAU)": "2.7 billion",
    "Videos indexed": "800 million",
    "Uploads per minute": "500 hours",
    "End-to-end latency budget": "200ms total",
    "Stage 1 target (retrieval)": "10ms - return 1000 candidates from 800M",
    "Stage 2 target (ranking)": "50ms - score 1000 candidates, return top 20",
    "Embedding dimension": "256 (user tower + video tower)",
    "ANN index type": "HNSW (FAISS) or ScaNN",
    "Retraining frequency": "Daily for ranking, weekly for retrieval",
    "Training data size": "~100B impression/click events per day",
}

print("=== YouTube Recommendations: Key Design Numbers ===\n")
for k, v in design_numbers.items():
    print(f"  {k:<45} {v}")

## 2. Two-Stage Architecture with Numbers

### Stage 1: Retrieval

**User tower inputs**: user ID embed (256-dim) + avg of last 50 watched video embeddings + device/locale/time categorical embeds
**Video tower inputs**: video ID embed (256-dim) + title+description text embed + category/channel embed + log(hours_since_upload)
**Training**: positives = (user, video) where user watched >50%; negatives = in-batch negatives; loss = InfoNCE/softmax
**ANN index**: HNSW, 800M videos x 256 dims x 4 bytes = ~800GB, sharded; retrieval in <10ms

### Stage 2: Ranking

**Features**: user-video interaction history, video quality signals (like rate, completion, report rate), context (time, device, session length), cross features
**Model**: Neural network distillation (large teacher -> small student for low latency)
**Training signal**: weighted combo: `0.4*(watch_pct >= 0.8) + 0.3*liked + 0.2*shared + 0.1*clicked - 0.5*disliked`

In [ ]:
stages = {
    "User embedding lookup (Redis)": 2,
    "User tower forward pass (GPU)": 3,
    "ANN index query (HNSW, 1000 results)": 8,
    "Feature retrieval for 1000 candidates": 15,
    "Ranking model forward pass (1000 items)": 25,
    "Business rules + diversity filter": 5,
    "Response serialization + network": 15,
}

total = sum(stages.values())
print("=== End-to-End Latency Budget ===")
print(f"{"Stage":<45} {"ms":>6}")
print("-" * 52)
for stage, ms in stages.items():
    bar = chr(9608) * int(ms / 2)
    print(f"{stage:<45} {ms:>4}ms {bar}")
print("-" * 52)
print(f"{"TOTAL":<45} {total:>4}ms")
print(f"\nBudget: 200ms | Used: {total}ms | Headroom: {200 - total}ms")
print("Note: P99 latency matters, not P50. In practice add 2-3x safety margin.")

## 3. Monitoring Plan

**What to track**:
1. **Business metrics** (daily): watch time/session, 7-day retention, DAU/MAU
2. **Model health** (hourly): NDCG@10 on random held-out sample with delayed labels
3. **Feature drift** (hourly): PSI on key user and video features
4. **Prediction distribution** (real-time): mean score, score variance - sudden shifts = bug
5. **Diversity** (daily): category distribution, new vs familiar creator ratio
6. **Cold start** (daily): recall@10 for videos < 24 hours old

**Rollback triggers**:
- AUC drops >5% vs baseline on held-out set
- Watch time/session drops >2% in A/B test (statistically significant)
- Any production error rate spike in feature pipeline

**Retraining**:
- Ranking model: daily retrain on 30-day rolling window
- Retrieval model: weekly retrain (more expensive; smaller embedding changes)

## 4. Interview Anti-Patterns

**Anti-pattern 1: Over-engineering Stage 1**
'I'll use a BERT model to encode every video in real-time.' BERT on 800M videos = impossible. Use precomputed embeddings + ANN index.

**Anti-pattern 2: Ignoring baselines**
Jumping to two-tower + transformer without mentioning: 'First, I'd establish a baseline - most popular videos filtered by watch history. This achieves X% of the value at 1% of the complexity.'

**Anti-pattern 3: Skipping monitoring**
'The system is done when it's deployed.' A design without monitoring is a design that will fail silently.

**Anti-pattern 4: Single metric optimization**
'I'll optimize CTR.' Always acknowledge the multi-objective nature and the tension between short-term and long-term engagement.

**Anti-pattern 5: Not asking clarifying questions**
The first 5 minutes should be: What's the latency budget? How many users? What's the existing baseline? What's most important to get right?

In [ ]:
topics = [
    ("Recommendation System", "Two-stage: two-tower retrieval -> neural ranker. Cold start, implicit feedback, diversity."),
    ("Search/Ranking", "BM25 baseline -> dense retrieval -> hybrid+RRF -> LTR (LambdaMART). Interleaving for A/B."),
    ("Feed Ranking", "Multi-objective scoring, IPS for position bias, freshness decay, creator Gini."),
    ("Ad CTR Prediction", "Wide & Deep, calibration (Platt scaling), delayed feedback, auction mechanics."),
    ("Fraud Detection", "GBM + threshold tuning, class imbalance, velocity features, concept drift retraining."),
    ("Content Moderation", "Latency tiers, confidence routing, multi-modal, adversarial robustness."),
    ("Entity Resolution", "Blocking (MinHash LSH), pairwise similarity, supervised deduplication."),
    ("Spam Detection", "TF-IDF + LR baseline, sender reputation, velocity features."),
    ("ETA / Time Prediction", "Spatial features, real-time traffic, confidence intervals, fallback to rules."),
    ("Abuse/Bot Detection", "Behavioral features, graph features (connected accounts), velocity, adversarial."),
]

print("=== Top 10 ML System Design Interview Topics ===\n")
for i, (topic, key_points) in enumerate(topics, 1):
    print(f"{i:2}. {topic}")
    print(f"    {key_points}")
    print()

## 5. Common Follow-Up Questions and Answers

**'How do you handle cold start?'** New users -> onboarding preferences + popular items + content-based fallback. New items -> content tower (metadata, audio/visual features).

**'How do you scale to 10x users?'** ANN index: shard by video category. Model: distillation for smaller, faster ranker. Feature store: Redis Cluster with read replicas.

**'What if the model recommends polarizing content?'** Filter bubble problem. Add diversity constraint to ranking objective. Monitor viewership concentration on political content; alert if trending up. Serve 5-10% from exploration policy.

**'How would you A/B test this?'** Holdout user cohort (10%). Measure watch time/session, 7-day retention, engagement breadth. Run 2 weeks minimum. Use sequential testing to stop early if results are clear.

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Start with constraints | Derive architecture from scale/latency requirements, not preferences |
| Baseline first | Simple rule-based or popularity baseline before neural models |
| Two-stage necessity | Can't score 800M videos per user request; ANN retrieval is mandatory |
| Monitoring = part of design | Drift detection, retraining triggers, rollback plan in every design |
| Anti-patterns | Over-engineering Stage 1, single metric, ignoring baselines, skipping monitoring |

### The Framework (One More Time)

1. **Problem framing**: business metric -> ML metric -> constraints
2. **Data**: sources, labels, volume, quality, feedback loop
3. **Features**: user, item, context, cross features; freshness requirements
4. **Model**: baseline first, then complexity justified by constraints
5. **Serving**: latency budget, ANN for retrieval, feature store for real-time features
6. **Monitoring**: feature drift, prediction drift, retraining triggers, rollback plan

### Common Pitfalls
- Designing the model before understanding the constraints
- Not mentioning baselines before proposing complex models
- Proposing a design without discussing failure modes
- Ignoring cold start in any recommendation/ranking system
- Not asking clarifying questions at the start
